

# Statistical Analyses:


1. Survival analysis: Time-to-event modeling for NT→NZ conversion. Cox proportional hazards to identify what predicts faster conversion.

2. Markov chains: Model state transitions (No commitment → NT:C → NT:T → NT+NZ). Calculate transition probabilities, steady-state distributions, expected time in each state.

3. Logistic regression: Predict which companies will convert NT→NZ based on cohort year, sector, region, initial status type.

4. Clustering: Group companies by their trajectory patterns (fast adopters, slow movers, dropouts, leapfroggers).

5. Churn analysis: Model why companies lose commitments (those 109 NZ losses in 2024→2025).

#### Temporal Analyses

6. Acceleration metrics: Is adoption speeding up? Compare slopes between cohorts.

7. Momentum indicators: Leading vs lagging sectors/regions in adoption waves.

8. Seasonality: Do commitments cluster around specific times (COP meetings, reporting cycles)?

#### Network/Portfolio Analyses

9. Portfolio risk: If X% typically drop targets, what's the expected stable state?


10. Contagion effects: If you had company relationships, model peer influence on adoption.

11. Optimal pathway: Which progression sequence has highest retention? (Direct to both vs stepwise)

#### Predictive Models

12. Time series forecasting: Project 2026-2030 adoption rates using ARIMA or exponential smoothing.

13. Cohort retention curves: Kaplan-Meier style plots showing retention by entry year.

14. Propensity scoring: Given attributes, probability of NT→NZ within 1/2/3 years.

#### What Would Be Most Insightful?

Given data quality, I'd prioritize:
- Markov chain model (clean state transitions, interpretable probabilities)
- Survival analysis (directly answers "when will they convert")
- Churn analysis (explains the anomalous NZ losses)





## What correlations to test:

1. Carbon credit usage vs commitment types
   - Companies saying they'll use carbon credits → higher likelihood of having CN/NZ/SBT?
   - Does CC usage correlate with faster NT→NZ conversion?

2. Commitment co-occurrence
   - If you have SBT, how likely to also have NZ?
   - If you have CN, how likely to mention CC usage?
   - Which commitments cluster together?

3. Temporal patterns
   - Does CC usage percentage change over time?
   - Does CC acceptance correlate with cohort year?

4. Regional/sectoral patterns
   - Which regions/sectors more likely to use CCs?
   - Does this correlate with commitment types?

## Statistical tests:

- Chi-square test: Independence between categorical variables (CC yes/no × SBT yes/no)
- Cramér's V: Strength of association (0-1 scale)
- Phi coefficient: For 2×2 tables specifically
- Point-biserial correlation: Binary (CC yes/no) vs continuous (number of commitments)
- Tetrachoric correlation: Underlying continuous relationship between two binary variables

## Example output:

"Companies using carbon credits are 2.3x more likely to have NZ targets (χ²=45.3, p<0.001, Cramér's V=0.28)"


## Markov 


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

df = pd.read_excel('historic_new.xlsx', sheet_name='sbti evolution ')
df.columns = df.columns.str.strip()
df['company'] = df['company'].astype(str)

years = ['2021', '2022', '2023', '2024', '2025']

# Define states
def get_state(nt, nz):
    nt = str(nt) if pd.notna(nt) else 'None'
    nz = str(nz) if pd.notna(nz) else 'None'
    
    if nt == 'None' and nz == 'None':
        return 'No commitment'
    if nt != 'None' and nz == 'None':
        return f'NT:{nt}'
    if nt == 'None' and nz != 'None':
        return f'NZ:{nz}'
    return f'NT:{nt}+NZ:{nz}'

# Build states for each company-year
states = {}
for _, row in df.iterrows():
    company = row['company']
    states[company] = {}
    for year in years:
        nt = row[f'{year}_NT_Status']
        nz = row[f'{year}_NZ_Status']
        states[company][year] = get_state(nt, nz)

# Count transitions
transitions = {}
for company in states:
    for i in range(len(years) - 1):
        from_state = states[company][years[i]]
        to_state = states[company][years[i+1]]
        
        if from_state not in transitions:
            transitions[from_state] = {}
        if to_state not in transitions[from_state]:
            transitions[from_state][to_state] = 0
        
        transitions[from_state][to_state] += 1

# Get all unique states
all_states = sorted(set(s for company in states.values() for s in company.values()))

# Build transition matrix
n = len(all_states)
matrix = np.zeros((n, n))
state_to_idx = {s: i for i, s in enumerate(all_states)}

for from_state in transitions:
    from_idx = state_to_idx[from_state]
    row_sum = sum(transitions[from_state].values())
    
    for to_state, count in transitions[from_state].items():
        to_idx = state_to_idx[to_state]
        matrix[from_idx, to_idx] = count / row_sum

# Print transition matrix
print("TRANSITION PROBABILITY MATRIX")
print("Rows = current state, Columns = next state\n")

# Print header
print(f"{'From State':<20}", end="")
for state in all_states:
    print(f"{state:<20}", end="")
print()

# Print matrix
for i, from_state in enumerate(all_states):
    print(f"{from_state:<20}", end="")
    for j in range(n):
        if matrix[i, j] > 0:
            print(f"{matrix[i, j]:.3f}              ", end="")
        else:
            print(f"{'.':<20}", end="")
    print()

# Steady state (eigenvector for eigenvalue 1)
eigenvalues, eigenvectors = np.linalg.eig(matrix.T)
steady_idx = np.argmax(np.abs(eigenvalues - 1.0) < 1e-10)
steady = np.real(eigenvectors[:, steady_idx])
steady = steady / steady.sum()

print("\n\nSTEADY STATE DISTRIBUTION")
print("Long-run equilibrium probabilities:\n")
for i, state in enumerate(all_states):
    print(f"{state:<30} {steady[i]:.3f} ({steady[i]*100:.1f}%)")

# Key transitions
print("\n\nKEY TRANSITIONS (>5% probability)")
key = []
for i, from_state in enumerate(all_states):
    for j, to_state in enumerate(all_states):
        if matrix[i, j] > 0.05 and i != j:
            key.append((from_state, to_state, matrix[i, j]))

key.sort(key=lambda x: -x[2])
for from_s, to_s, prob in key:
    print(f"{from_s:<25} → {to_s:<25} {prob:.3f}")

# Visualization: Sankey of top transitions
sources = []
targets = []
values = []
labels = all_states.copy()

for i, from_state in enumerate(all_states):
    for j, to_state in enumerate(all_states):
        if matrix[i, j] > 0.02:
            sources.append(i)
            targets.append(j)
            values.append(matrix[i, j])

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        label=labels,
        color='lightblue'
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values
    )
)])

fig.update_layout(
    title="Markov Chain Transition Probabilities (edges > 2%)",
    height=800,
    width=1200
)

import os
os.chdir('/mnt/user-data/outputs')
fig.write_html('markov_transitions.html')

# Save transition matrix
tm_df = pd.DataFrame(matrix, index=all_states, columns=all_states)
tm_df.to_csv('transition_matrix.csv')

# Save steady state
ss_df = pd.DataFrame({'state': all_states, 'probability': steady})
ss_df.to_csv('steady_state.csv', index=False)

print("\n\nFiles: markov_transitions.html, transition_matrix.csv, steady_state.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'historic_new.xlsx'

## Cross-sectional analyses (2025 only):

Chi-square independence tests: CC usage vs NZ, CC vs SBT, etc.
Cramér's V correlation matrix: Heatmap of all commitment associations
Conditional probabilities: P(NZ | SBT), P(CC | CN), etc.
Logistic regression: Predict NZ from sector, region, CC, NT status
Cluster analysis: Group companies by commitment profile
Sector/region benchmarking: Which sectors lead in each commitment type

## Longitudinal with 2024-2025:

Year-over-year changes: Who gained/lost each commitment
Transition analysis: 2024 state → 2025 state (9x9 matrix)
Upgrade/downgrade rates: C→T vs T→C vs dropouts
Logistic for change: Predict who converts/drops based on 2024 profile

# 2025 only stats


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
import plotly.graph_objects as go

df = pd.read_excel(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\historic new .xlsx', sheet_name='2025', header=1, nrows=500)
df.columns = ['company', 're100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no', 'any_action']
df = df.drop(0).reset_index(drop=True)

for col in ['re100', 'has_nt', 'gov_cn', 'cn', 'nz', 'cc_yes', 'cc_no', 'any_action']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

vars_test = ['has_nt', 'nz', 'cc_yes', 'cn', 're100']

print("CHI-SQUARE")
for i, v1 in enumerate(vars_test):
    for v2 in vars_test[i+1:]:
        ct = pd.crosstab(df[v1], df[v2])
        chi2, p, _, _ = chi2_contingency(ct)
        v = np.sqrt(chi2 / (ct.sum().sum() * (min(ct.shape) - 1)))
        print(f"{v1} vs {v2}: V={v:.3f}, p={p:.4f}")

print("\nCONDITIONAL PROB")
print(f"P(nz|has_nt) = {df.loc[df['has_nt']==1, 'nz'].mean():.3f}")
print(f"P(nz|cc_yes) = {df.loc[df['cc_yes']==1, 'nz'].mean():.3f}")
print(f"P(cc_yes|cn) = {df.loc[df['cn']==1, 'cc_yes'].mean():.3f}")
print(f"P(has_nt|re100) = {df.loc[df['re100']==1, 'has_nt'].mean():.3f}")

print("\nLOGISTIC: PREDICT NZ")
X = df[['has_nt', 'cc_yes', 'cn', 're100', 'nz']]
lr = LogisticRegression(max_iter=1000)
lr.fit(X, df['nz'])
for feat, coef in zip(X.columns, lr.coef_[0]):
    print(f"{feat}: OR={np.exp(coef):.2f}")
print("\nLOGISTIC: PREDICT CC")
lr.fit(X, df['cc_yes'])
for feat, coef in zip(X.columns, lr.coef_[0]):
    print(f"{feat}: OR={np.exp(coef):.2f}")

print("\nCLUSTERS")
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(df[vars_test])
for i in range(4):
    n = (df['cluster']==i).sum()
    means = df[df['cluster']==i][vars_test].mean()
    print(f"C{i} (n={n}): nt={means['has_nt']:.2f}, nz={means['nz']:.2f}, cc={means['cc_yes']:.2f}, cn={means['cn']:.2f}, re={means['re100']:.2f}")

n = len(vars_test)
corr = np.zeros((n, n))
for i, v1 in enumerate(vars_test):
    for j, v2 in enumerate(vars_test):
        if i == j:
            corr[i,j] = 1.0
        else:
            ct = pd.crosstab(df[v1], df[v2])
            chi2, _, _, _ = chi2_contingency(ct)
            corr[i,j] = np.sqrt(chi2 / (ct.sum().sum() * (min(ct.shape) - 1)))

print("\nCRAMERS V")
print(pd.DataFrame(corr, index=vars_test, columns=vars_test).round(3))

fig = go.Figure(go.Heatmap(z=corr, x=vars_test, y=vars_test, colorscale='Blues', text=np.round(corr, 3), texttemplate='%{text}'))
fig.update_layout(title="Cramers V", height=600, width=700)
fig.write_html('cramers_v.html')




CHI-SQUARE
has_nt vs nz: V=0.309, p=0.0000
has_nt vs cc_yes: V=0.099, p=0.0265
has_nt vs cn: V=0.044, p=0.3234
has_nt vs re100: V=0.280, p=0.0000
nz vs cc_yes: V=0.435, p=0.0000
nz vs cn: V=0.472, p=0.0000
nz vs re100: V=0.284, p=0.0000
cc_yes vs cn: V=0.040, p=0.3748
cc_yes vs re100: V=0.150, p=0.0008
cn vs re100: V=0.065, p=0.1449

CONDITIONAL PROB
P(nz|has_nt) = 0.737
P(nz|cc_yes) = 0.751
P(cc_yes|cn) = 0.396
P(has_nt|re100) = 0.623

LOGISTIC: PREDICT NZ
has_nt: OR=1.79
cc_yes: OR=2.44
cn: OR=0.26
re100: OR=1.66
nz: OR=628.93

LOGISTIC: PREDICT CC
has_nt: OR=0.98
cc_yes: OR=954.85
cn: OR=1.37
re100: OR=1.13
nz: OR=2.35

CLUSTERS
C0 (n=88): nt=0.60, nz=0.98, cc=0.00, cn=0.00, re=0.28
C1 (n=171): nt=0.40, nz=0.97, cc=1.00, cn=0.00, re=0.25
C2 (n=91): nt=0.26, nz=0.00, cc=0.40, cn=1.00, re=0.10
C3 (n=149): nt=0.07, nz=0.00, cc=0.09, cn=0.00, re=0.00

CRAMERS V
        has_nt     nz  cc_yes     cn  re100
has_nt   1.000  0.309   0.099  0.044  0.280
nz       0.309  1.000   0.435  0.472  0

In [4]:
print(pd.DataFrame(corr, index=vars_test, columns=vars_test).round(3))

        has_nt     nz  cc_yes     cn  re100
has_nt   1.000  0.309   0.099  0.044  0.280
nz       0.309  1.000   0.435  0.472  0.284
cc_yes   0.099  0.435   1.000  0.040  0.150
cn       0.044  0.472   0.040  1.000  0.065
re100    0.280  0.284   0.150  0.065  1.000


## overlap matrix

In [5]:
# List of key variables
vars_summary = ['nz', 'cn', 're100', 'has_nt', 'cc_yes']

# Initialize empty DataFrame for the overlap counts
overlap_matrix = pd.DataFrame(index=vars_summary, columns=vars_summary)

# Fill the matrix
for row_var in vars_summary:
    for col_var in vars_summary:
        overlap_matrix.loc[row_var, col_var] = ((df[row_var]==1) & (df[col_var]==1)).sum()

# Convert to integer
overlap_matrix = overlap_matrix.astype(int)

print(overlap_matrix)


         nz  cn  re100  has_nt  cc_yes
nz      252   0     65     115     166
cn        0  91      9      24      36
re100    65   9     77      48      48
has_nt  115  24     48     156      81
cc_yes  166  36     48      81     221


## Further Analysis 2025-2024

In [ ]:
print("ANALYSIS 1: CHI-SQUARE INDEPENDENCE TESTS (2025)")


vars_2025 = ['has_nt_2025', 'has_nz_2025', 'cc_usage', 'cn', 're100']
chi_results = []

for i, var1 in enumerate(vars_2025):
    for var2 in vars_2025[i+1:]:
        ct = pd.crosstab(df[var1], df[var2])
        chi2, p, dof, exp = chi2_contingency(ct)
        n = ct.sum().sum()
        cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
        chi_results.append({
            'var1': var1,
            'var2': var2,
            'chi2': chi2,
            'p': p,
            'cramers_v': cramers_v
        })
        print(f"{var1} vs {var2}: chi2={chi2:.2f}, p={p:.4f}, V={cramers_v:.3f}")

chi_df = pd.DataFrame(chi_results)

In [ ]:

print("\nANALYSIS 2: CONDITIONAL PROBABILITIES (2025)")


conditions = [
    ('has_nz_2025', 'has_nt_2025'),
    ('has_nz_2025', 'cc_usage'),
    ('cc_usage', 'cn'),
    ('has_nt_2025', 're100')
]

for outcome, given in conditions:
    prob = df[df[given]==1][outcome].mean()
    print(f"P({outcome} | {given}=1) = {prob:.3f}")

In [ ]:


print("\nANALYSIS 3: YEAR-OVER-YEAR CHANGES")


for var in ['has_nt', 'has_nz']:
    gained = ((df[f'{var}_2024']==0) & (df[f'{var}_2025']==1)).sum()
    lost = ((df[f'{var}_2024']==1) & (df[f'{var}_2025']==0)).sum()
    kept = ((df[f'{var}_2024']==1) & (df[f'{var}_2025']==1)).sum()
    print(f"{var}: +{gained} gained, -{lost} lost, {kept} kept")


In [ ]:

print("\nANALYSIS 4: STATE TRANSITIONS 2024->2025")


def state(row, year):
    nt = row[f'{year}_NT_Status']
    nz = row[f'{year}_NZ_Status']
    if pd.isna(nt) and pd.isna(nz): return 'None'
    if pd.notna(nt) and pd.isna(nz): return 'NT'
    if pd.isna(nt) and pd.notna(nz): return 'NZ'
    return 'Both'

df['state_2024'] = df.apply(lambda r: state(r, '2024'), axis=1)
df['state_2025'] = df.apply(lambda r: state(r, '2025'), axis=1)

trans_ct = pd.crosstab(df['state_2024'], df['state_2025'])
print(trans_ct)



## Logistic regression 2025

In [9]:
print("\nANALYSIS 5: LOGISTIC REGRESSION - PREDICT NZ_2025")


X = df[['has_nt', 'cc_yes', 'cn', 're100']].fillna(0)
y = df['nz']

lr = LogisticRegression(max_iter=1000)
lr.fit(X, y)

for feat, coef in zip(X.columns, lr.coef_[0]):
    odds_ratio = np.exp(coef)
    print(f"{feat}: coef={coef:.3f}, OR={odds_ratio:.3f}")





ANALYSIS 5: LOGISTIC REGRESSION - PREDICT NZ_2025
has_nt: coef=1.550, OR=4.709
cc_yes: coef=2.310, OR=10.071
cn: coef=-4.504, OR=0.011
re100: coef=1.351, OR=3.861


In [10]:
print("\nANALYSIS 5: LOGISTIC REGRESSION - PREDICT CC_2025")


X = df[['has_nt', 'nz', 'cn', 're100']].fillna(0)
y = df['cc_yes']

lr = LogisticRegression(max_iter=1000)
lr.fit(X, y)

for feat, coef in zip(X.columns, lr.coef_[0]):
    odds_ratio = np.exp(coef)
    print(f"{feat}: coef={coef:.3f}, OR={odds_ratio:.3f}")



ANALYSIS 5: LOGISTIC REGRESSION - PREDICT CC_2025
has_nt: coef=-0.284, OR=0.753
nz: coef=2.438, OR=11.445
cn: coef=1.311, OR=3.710
re100: coef=0.204, OR=1.227


In [11]:
import pandas as pd

# Check the actual counts
print("CROSSTAB: NZ vs CC_YES")
ct = pd.crosstab(df['nz'], df['cc_yes'], margins=True)
print(ct)

print("\n\nMANUAL ODDS CALCULATION:")

# Companies WITH NZ
nz_yes = df[df['nz']==1]
nz_has_cc = (nz_yes['cc_yes']==1).sum()
nz_no_cc = (nz_yes['cc_yes']==0).sum()
odds_nz = nz_has_cc / nz_no_cc if nz_no_cc > 0 else float('inf')
print(f"NZ companies: {nz_has_cc} have CC, {nz_no_cc} don't have CC")
print(f"Odds of CC given NZ: {odds_nz:.3f}")

# Companies WITHOUT NZ
nz_no = df[df['nz']==0]
no_nz_has_cc = (nz_no['cc_yes']==1).sum()
no_nz_no_cc = (nz_no['cc_yes']==0).sum()
odds_no_nz = no_nz_has_cc / no_nz_no_cc if no_nz_no_cc > 0 else 0
print(f"\nNo NZ companies: {no_nz_has_cc} have CC, {no_nz_no_cc} don't have CC")
print(f"Odds of CC given no NZ: {odds_no_nz:.3f}")

# Odds Ratio
if odds_no_nz > 0:
    or_manual = odds_nz / odds_no_nz
    print(f"\n**Odds Ratio (manual): {or_manual:.3f}**")
    print(f"Companies with NZ are {or_manual:.1f}x more likely to use CC")
else:
    print("\nCannot calculate OR (division by zero)")

# Compare to logistic regression result
print(f"\nLogistic Regression OR: 11.445")

CROSSTAB: NZ vs CC_YES
cc_yes    0    1  All
nz                   
0       192   55  247
1        86  166  252
All     278  221  499


MANUAL ODDS CALCULATION:
NZ companies: 166 have CC, 86 don't have CC
Odds of CC given NZ: 1.930

No NZ companies: 55 have CC, 192 don't have CC
Odds of CC given no NZ: 0.286

**Odds Ratio (manual): 6.738**
Companies with NZ are 6.7x more likely to use CC

Logistic Regression OR: 11.445


 "When you compare two companies with identical NT, CN, and RE100 status, the one with NZ is 11.4x more likely to use credits"

The simple odds ratio (6.7x) mixes in confounding effects - some of that NZ→CC association might be because NZ companies also have NT, RE100, etc.

Conservative claim: "Companies with net zero are 7x more likely to use carbon credits" (manual OR)
Controlled claim: "Net zero independently predicts 11x higher credit usage, even accounting for other climate commitments" (logistic OR)

The 6.7x is easier to verify and more intuitive. The 11.4x is technically more rigorous but harder to explain.

In [12]:
print("\nANALYSIS 6: SECTOR/REGION BENCHMARKING (2025)")


for group in ['sector', 'region']:
    print(f"\n{group}:")
    agg = df.groupby(group)[['has_nt', 'nz', 'cc_yes']].mean()
    print(agg.round(3))



ANALYSIS 6: SECTOR/REGION BENCHMARKING (2025)

sector:


KeyError: 'sector'

In [ ]:
print("\nANALYSIS 7: CLUSTER ANALYSIS (2025)")


X_cluster = df[['has_nt_2025', 'has_nz_2025', 'cc_usage', 'cn', 're100']].fillna(0)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_cluster)

for i in range(4):
    cluster_df = df[df['cluster']==i]
    n = len(cluster_df)
    profile = cluster_df[['has_nt_2025', 'has_nz_2025', 'cc_usage', 'cn', 're100']].mean()
    print(f"\nCluster {i} (n={n}):")
    print(profile.round(3).to_dict())


In [ ]:
print("\nANALYSIS 8: UPGRADE/DOWNGRADE RATES")


df['nt_upgrade'] = ((df['2024_NT_Status']=='C') & (df['2025_NT_Status']=='T')).astype(int)
df['nt_downgrade'] = ((df['2024_NT_Status']=='T') & (df['2025_NT_Status']=='C')).astype(int)
df['nz_upgrade'] = ((df['2024_NZ_Status']=='C') & (df['2025_NZ_Status']=='T')).astype(int)
df['nz_downgrade'] = ((df['2024_NZ_Status']=='T') & (df['2025_NZ_Status']=='C')).astype(int)

print(f"NT upgrades: {df['nt_upgrade'].sum()}")
print(f"NT downgrades: {df['nt_downgrade'].sum()}")
print(f"NZ upgrades: {df['nz_upgrade'].sum()}")
print(f"NZ downgrades: {df['nz_downgrade'].sum()}")

In [ ]:
print("\nANALYSIS 9: LOGISTIC FOR CHANGE - WHO CONVERTS 2024->2025")


converters = df[(df['has_nz_2024']==0) & (df['has_nz_2025']==1)]
non_converters = df[(df['has_nz_2024']==0) & (df['has_nz_2025']==0)]
subset = pd.concat([converters, non_converters])

X_change = subset[['has_nt_2024', 'cc_usage', 'cn', 're100']].fillna(0)
y_change = (subset['has_nz_2025']==1).astype(int)

if len(y_change.unique()) > 1:
    lr_change = LogisticRegression(max_iter=1000)
    lr_change.fit(X_change, y_change)
    
    print("Predictors of NZ adoption (among non-NZ in 2024):")
    for feat, coef in zip(X_change.columns, lr_change.coef_[0]):
        odds_ratio = np.exp(coef)
        print(f"{feat}: OR={odds_ratio:.3f}")


In [ ]:
print("\nANALYSIS 10: CRAMERS V CORRELATION MATRIX 2025")


vars_corr = ['has_nt', 'nz', 'cc_yes', 'cn', 're100']
n_vars = len(vars_corr)
corr_matrix = np.zeros((n_vars, n_vars))

for i, v1 in enumerate(vars_corr):
    for j, v2 in enumerate(vars_corr):
        if i == j:
            corr_matrix[i, j] = 1.0
        else:
            ct = pd.crosstab(df[v1], df[v2])
            chi2, p, dof, exp = chi2_contingency(ct)
            n = ct.sum().sum()
            cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
            corr_matrix[i, j] = cramers_v

corr_df = pd.DataFrame(corr_matrix, index=vars_corr, columns=vars_corr)
print(corr_df.round(3))


ANALYSIS 10: CRAMERS V CORRELATION MATRIX
        has_nt     nz  cc_yes     cn  re100
has_nt   1.000  0.309   0.099  0.044  0.280
nz       0.309  1.000   0.435  0.472  0.284
cc_yes   0.099  0.435   1.000  0.040  0.150
cn       0.044  0.472   0.040  1.000  0.065
re100    0.280  0.284   0.150  0.065  1.000


In [19]:
import pandas as pd
import openpyxl

file_path = r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new.xlsx'

name_mappings = {
    'AmerisourceBergen': 'Cencora',
    'Amer International Group': 'Cencora',
    'Electricité de France': 'Electricite de France',
    'Deutsche Post DHL Group': 'DHL Group',
    'Nippon Telegraph and Telephone': 'NTT',
    'Brookfield Asset Management': 'Brookfield',
    'SK Group': 'SK',
    'General Electric': 'General Electric (GE Aerospace)',
    'Bunge': 'Bunge Global',
    'POSCO': 'POSCO Holdings',
    'PKN ORLEN Group': 'Orlen',
    'América Móvil': 'America Movil',
    'Raízen': 'Raizen',
    'Mitsubishi Corp': 'Mitsubishi',
    'International Business Machines': 'IBM',
    'Raytheon Technologies': 'RTX',
    'World Fuel Services': 'World Kinect',
    'Anthem': 'Elevance Health',
    'Daimler': 'Mercedes-Benz Group',
    'Facebook': 'Meta Platforms',
    'Royal Dutch Shell': 'Shell',
    'Sinochem': 'Sinochem Holdings',
    'Panasonic': 'Panasonic Holdings',
    'GlaxoSmithKline': 'GSK',
    'ViacomCBS': 'Paramount Global',
    'Synnex': 'TD Synnex',
}

wb = openpyxl.load_workbook(file_path)

for sheet in wb.worksheets:
    for row in sheet.iter_rows():
        for cell in row:
            if cell.value in name_mappings:
                cell.value = name_mappings[cell.value]

wb.save(file_path)

In [6]:
import pandas as pd

file_path = r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\progression anlysis data.xlsx'
portfolio = pd.read_excel(file_path, header=0)

def normalize(val):
    if pd.isna(val):
        return '-'
    if val == 0 or val == 'N':
        return '0'
    return '1'

commitment_cols = [c for c in portfolio.columns if isinstance(c, str) and any(x in c for x in ['_cn', '_nz', '_re', '_sbti', '_cc'])]
for col in commitment_cols:
    portfolio[col] = portfolio[col].apply(normalize)

portfolio['cn_progression'] = portfolio[['2021_cn', '2022_cn', '2023_cn', '2024_cn', '2025_cn']].apply(lambda x: ''.join(x), axis=1)
portfolio['nz_progression'] = portfolio[['2021_nz', '2022_nz', '2023_nz', '2024_nz', '2025_nz']].apply(lambda x: ''.join(x), axis=1)
portfolio['re_progression'] = portfolio[['2021_re', '2022_re', '2023_re', '2024_re', '2025_re']].apply(lambda x: ''.join(x), axis=1)
portfolio['sbti_progression'] = portfolio[['2021_sbti_nt', '2022_sbti_nt', '2023_sbti_nt', '2024_sbti_nt', '2025_sbti_nt']].apply(lambda x: ''.join(x), axis=1)

print("Carbon Neutral:")
print(portfolio['cn_progression'].value_counts().head(15))

print("\nNet Zero:")
print(portfolio['nz_progression'].value_counts().head(15))

print("\nRE100:")
print(portfolio['re_progression'].value_counts().head(15))

print("\nSBTi:")
print(portfolio['sbti_progression'].value_counts().head(15))

KeyError: "None of [Index(['2021_cn', '2022_cn', '2023_cn', '2024_cn', '2025_cn'], dtype='object')] are in the [columns]"

In [3]:
import pandas as pd

file_path = r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\progression anlysis data.xlsx'
portfolio = pd.read_excel(file_path, header=0)

print(portfolio.columns.tolist())
print(portfolio.shape)

['company', 'country', 'sector ', '2021 cc yes/no', '2022 cc yes/no', '2023 cc yes/no', '2024 cc yes/no', '2025 cc yes/no', '2021 cn ', '2022 cn ', '2023 cn ', '2024 cn ', '2025 cn ', '2021 re', '2022 re', '2023 re', '2024 re', '2025 re', '2021 sbti nt', '2022 sbti nt', '2023 sbti nt', '2024 sbti nt', '2025 sbti nt', '2021 nz', '2022 nz', '2023 nz', '2024 nz', '2025 nz']
(651, 28)


In [7]:
import pandas as pd

file_path = r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\progression anlysis data.xlsx'
portfolio = pd.read_excel(file_path, header=0)

def normalize(val):
    if pd.isna(val):
        return '-'
    if val == 0 or val == 'N':
        return '0'
    return '1'

cc_cols = ['2021 cc yes/no', '2022 cc yes/no', '2023 cc yes/no', '2024 cc yes/no', '2025 cc yes/no']
cn_cols = ['2021 cn ', '2022 cn ', '2023 cn ', '2024 cn ', '2025 cn ']
re_cols = ['2021 re', '2022 re', '2023 re', '2024 re', '2025 re']
sbti_cols = ['2021 sbti nt', '2022 sbti nt', '2023 sbti nt', '2024 sbti nt', '2025 sbti nt']
nz_cols = ['2021 nz', '2022 nz', '2023 nz', '2024 nz', '2025 nz']

for col in cc_cols + cn_cols + re_cols + sbti_cols + nz_cols:
    portfolio[col] = portfolio[col].apply(normalize)

portfolio['cc_progression'] = portfolio[cc_cols].apply(lambda x: ''.join(x), axis=1)
portfolio['cn_progression'] = portfolio[cn_cols].apply(lambda x: ''.join(x), axis=1)
portfolio['re_progression'] = portfolio[re_cols].apply(lambda x: ''.join(x), axis=1)
portfolio['sbti_progression'] = portfolio[sbti_cols].apply(lambda x: ''.join(x), axis=1)
portfolio['nz_progression'] = portfolio[nz_cols].apply(lambda x: ''.join(x), axis=1)

print("Carbon Credits:")
print(portfolio['cc_progression'].value_counts().head(15))
print("\nCarbon Neutral:")
print(portfolio['cn_progression'].value_counts().head(15))
print("\nNet Zero:")
print(portfolio['nz_progression'].value_counts().head(15))
print("\nRE100:")
print(portfolio['re_progression'].value_counts().head(15))
print("\nSBTi:")
print(portfolio['sbti_progression'].value_counts().head(15))

Carbon Credits:
cc_progression
--000    97
-----    64
--100    60
--111    38
-1111    29
--011    26
-1011    24
---00    22
--010    20
--101    20
--0--    19
----1    17
--001    17
--110    13
---11    11
Name: count, dtype: int64

Carbon Neutral:
cn_progression
----0    207
-----    102
----1    102
11--1     39
11111     22
11110     15
1---1     11
---11     10
11---     10
--111     10
1----      9
-1--0      9
-1--1      9
11--0      8
--110      7
Name: count, dtype: int64

Net Zero:
nz_progression
---00    203
-----     71
11111     67
-1111     58
---11     33
---01     24
---0-     23
--111     20
----1     15
----0     15
11---     12
---10     11
1----      9
-1---      9
1-111      9
Name: count, dtype: int64

RE100:
re_progression
00000    280
11111     39
0----     32
-0000     30
---00     27
00---     25
000--     25
--000     22
----0     21
0000-     18
-00--     12
---0-      8
1----      8
01111      8
---11      8
Name: count, dtype: int64

SBTi:
sbti_progres

In [8]:
# Companies that gained commitments (went from 0/- to 1)
portfolio['cn_gained'] = portfolio['cn_progression'].str.contains('0.*1|^-.*1')
portfolio['nz_gained'] = portfolio['nz_progression'].str.contains('0.*1|^-.*1')
portfolio['re_gained'] = portfolio['re_progression'].str.contains('0.*1|^-.*1')
portfolio['sbti_gained'] = portfolio['sbti_progression'].str.contains('0.*1|^-.*1')

# Companies that lost commitments (went from 1 to 0/-)
portfolio['cn_lost'] = portfolio['cn_progression'].str.contains('1.*0|1.*-')
portfolio['nz_lost'] = portfolio['nz_progression'].str.contains('1.*0|1.*-')
portfolio['re_lost'] = portfolio['re_progression'].str.contains('1.*0|1.*-')
portfolio['sbti_lost'] = portfolio['sbti_progression'].str.contains('1.*0|1.*-')

# Early adopters (had commitment in 2021)
portfolio['cn_early'] = portfolio['2021 cn '] == '1'
portfolio['nz_early'] = portfolio['2021 nz'] == '1'
portfolio['re_early'] = portfolio['2021 re'] == '1'
portfolio['sbti_early'] = portfolio['2021 sbti nt'] == '1'

print(f"CN: {portfolio['cn_gained'].sum()} gained, {portfolio['cn_lost'].sum()} lost, {portfolio['cn_early'].sum()} early adopters")
print(f"NZ: {portfolio['nz_gained'].sum()} gained, {portfolio['nz_lost'].sum()} lost, {portfolio['nz_early'].sum()} early adopters")
print(f"RE: {portfolio['re_gained'].sum()} gained, {portfolio['re_lost'].sum()} lost, {portfolio['re_early'].sum()} early adopters")
print(f"SBTi: {portfolio['sbti_gained'].sum()} gained, {portfolio['sbti_lost'].sum()} lost, {portfolio['sbti_early'].sum()} early adopters")

# Companies with all 5 commitments in 2025
portfolio['all_2025'] = (
    (portfolio['2025 cn '] == '1') & 
    (portfolio['2025 nz'] == '1') & 
    (portfolio['2025 re'] == '1') & 
    (portfolio['2025 sbti nt'] == '1') &
    (portfolio['2025 cc yes/no'] == '1')
)
print(f"\nCompanies with all 5 commitments in 2025: {portfolio['all_2025'].sum()}")

CN: 194 gained, 195 lost, 148 early adopters
NZ: 221 gained, 122 lost, 126 early adopters
RE: 41 gained, 41 lost, 74 early adopters
SBTi: 113 gained, 46 lost, 86 early adopters

Companies with all 5 commitments in 2025: 23


In [9]:
gained_nz = portfolio[portfolio['nz_gained']][['company', 'nz_progression', 'country', 'sector ']]
lost_nz = portfolio[portfolio['nz_lost']][['company', 'nz_progression', 'country', 'sector ']]

gained_sbti = portfolio[portfolio['sbti_gained']][['company', 'sbti_progression', 'country', 'sector ']]
lost_sbti = portfolio[portfolio['sbti_lost']][['company', 'sbti_progression', 'country', 'sector ']]

print(f"Net Zero - Gained: {len(gained_nz)}, Lost: {len(lost_nz)}")
print(gained_nz.head(20))
print("\n" + "="*80 + "\n")
print(lost_nz.head(20))

print("\n\n" + "="*80 + "\n\n")

print(f"SBTi - Gained: {len(gained_sbti)}, Lost: {len(lost_sbti)}")
print(gained_sbti.head(20))
print("\n" + "="*80 + "\n")
print(lost_sbti.head(20))

gained_nz.to_csv(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\nz_gained.csv', index=False)
lost_nz.to_csv(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\nz_lost.csv', index=False)
gained_sbti.to_csv(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\sbti_gained.csv', index=False)
lost_sbti.to_csv(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\sbti_lost.csv', index=False)

Net Zero - Gained: 221, Lost: 122
                    company nz_progression  country  sector 
1                       ABB          ---11      NaN      NaN
2                       ACS          ---01      NaN      NaN
4                 AIA Group          -1-11      NaN      NaN
5                       AIG          ---1-      NaN      NaN
6        ANZ Group Holdings          ---10      NaN      NaN
7                      AT&T          ---01      NaN      NaN
9                    AbbVie          ---10      NaN      NaN
10      Abbott Laboratories          ---01      NaN      NaN
13                    Aegon          -1---      NaN      NaN
15     Air France-KLM Group          ---11      NaN      NaN
16                   Airbus          -1110      NaN      NaN
18               Albertsons          --111      NaN      NaN
23                 Allstate          --111      NaN      NaN
24                 Alphabet          -1111      NaN      NaN
28            America Movil          ---01      NaN

In [13]:

portfolio['nz_lost'] = portfolio['nz_progression'].str.match(r'^[^1]*1[01-]*0[0-]*$', na=False)
portfolio['sbti_lost'] = portfolio['sbti_progression'].str.match(r'^[^1]*1[01-]*0[0-]*$', na=False)

lost_nz = portfolio[portfolio['nz_lost']][['company', 'nz_progression', 'country', 'sector ']]
lost_sbti = portfolio[portfolio['sbti_lost']][['company', 'sbti_progression', 'country', 'sector ']]

print(f"Net Zero lost: {len(lost_nz)}")
print(lost_nz)

print(f"\nSBTi lost: {len(lost_sbti)}")
print(lost_sbti)

lost_nz.to_csv(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\nz_lost.csv', index=False)
lost_sbti.to_csv(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\sbti_lost.csv', index=False)

Net Zero lost: 31
                                    company nz_progression  country  sector 
6                        ANZ Group Holdings          ---10      NaN      NaN
9                                    AbbVie          ---10      NaN      NaN
16                                   Airbus          -1110      NaN      NaN
17                                    Aisin          1--0-      NaN      NaN
41                                    Apple          1--00      NaN      NaN
53                              BNP Paribas          11-00      NaN      NaN
76                                   Boeing          11100      NaN      NaN
140                         China Minmetals          --100      NaN      NaN
144  China National Building Material Group          ---10      NaN      NaN
298                           Hyundai Motor          --100      NaN      NaN
337                                     Kia          ---10      NaN      NaN
365                                  Lukoil          ---10

In [14]:
lost_sbti_with_nz = portfolio[portfolio['sbti_lost']][['company', 'sbti_progression', 'nz_progression', 'nz_gained']]
print(f"Companies that lost SBTi and gained NZ: {lost_sbti_with_nz['nz_gained'].sum()}")
print(lost_sbti_with_nz)

Companies that lost SBTi and gained NZ: 8
                    company sbti_progression nz_progression  nz_gained
10      Abbott Laboratories            --110          ---01       True
28            America Movil            ---10          ---01       True
78                 Bouygues            ---10          ---01       True
92    CK Hutchison Holdings            --110          1-111      False
181              Coop Group            --1-0          -1111       True
213               ELO Group            --1-0          ---00      False
388                   Metro            11110          ---00      False
398                  Mitsui            -1--0          11101       True
425            Novo Nordisk            ---10          ---11       True
572             Tata Motors            -1--0          1-101       True
613  Verizon Communications            11110          -1111       True


ck hutchkinson is nz + so only two lost sbti and didnt gain nz

double check the - and 0 make sure - represents absence in that year and matches for all years for each company 

In [15]:
import plotly.graph_objects as go
import numpy as np

years = ['2021', '2022', '2023', '2024', '2025']
nodes = []
node_colors = []

for year in years:
    nodes.extend([f'{year} None', f'{year} SBTi only', f'{year} NZ only', f'{year} Both'])
    node_colors.extend(['#cccccc', '#008ACA', '#00B4B2', '#8B479B'])

source = []
target = []
value = []
link_colors = []

for i in range(len(years) - 1):
    current_year = years[i]
    next_year = years[i + 1]
    
    mask = (portfolio[f'{current_year} sbti nt'] != '-') & (portfolio[f'{next_year} sbti nt'] != '-') & \
           (portfolio[f'{current_year} nz'] != '-') & (portfolio[f'{next_year} nz'] != '-')
    subset = portfolio[mask]
    
    transitions = subset.groupby([f'{current_year} sbti nt', f'{current_year} nz', 
                                   f'{next_year} sbti nt', f'{next_year} nz']).size().reset_index(name='count')
    
    for _, row in transitions.iterrows():
        sbti_curr = int(row[f'{current_year} sbti nt'])
        nz_curr = int(row[f'{current_year} nz'])
        sbti_next = int(row[f'{next_year} sbti nt'])
        nz_next = int(row[f'{next_year} nz'])
        
        if sbti_curr == 0 and nz_curr == 0:
            src_state = 0
        elif sbti_curr == 1 and nz_curr == 0:
            src_state = 1
        elif sbti_curr == 0 and nz_curr == 1:
            src_state = 2
        else:
            src_state = 3
            
        if sbti_next == 0 and nz_next == 0:
            tgt_state = 0
        elif sbti_next == 1 and nz_next == 0:
            tgt_state = 1
        elif sbti_next == 0 and nz_next == 1:
            tgt_state = 2
        else:
            tgt_state = 3
        
        src_idx = i * 4 + src_state
        tgt_idx = (i + 1) * 4 + tgt_state
        
        source.append(src_idx)
        target.append(tgt_idx)
        value.append(row['count'])
        
        src_color = node_colors[src_idx]
        tgt_color = node_colors[tgt_idx]
        link_colors.append(f'rgba({int(src_color[1:3], 16)}, {int(src_color[3:5], 16)}, {int(src_color[5:7], 16)}, 0.3)')

fig = go.Figure(data=[go.Sankey(
    node=dict(pad=15, thickness=20, label=nodes, color=node_colors),
    link=dict(source=source, target=target, value=value, color=link_colors)
)])

fig.update_layout(title_text="SBTi & Net Zero Transitions 2021-2025", height=800)
fig.write_html(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\sbti_nz_sankey.html')
fig.show()

In [16]:
import pandas as pd

file_path = r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\progression anlysis data.xlsx'
portfolio = pd.read_excel(file_path, header=0)

def normalize(val):
    if pd.isna(val):
        return '-'
    if val == 0 or val == 'N':
        return '0'
    return '1'

sbti_cols = ['2021 sbti nt', '2022 sbti nt', '2023 sbti nt', '2024 sbti nt', '2025 sbti nt']
nz_cols = ['2021 nz', '2022 nz', '2023 nz', '2024 nz', '2025 nz']

for col in sbti_cols + nz_cols:
    portfolio[col] = portfolio[col].apply(normalize)

transitions = []
for idx, row in portfolio.iterrows():
    first_sbti_year = None
    first_nz_year = None
    
    for i, year in enumerate(['2021', '2022', '2023', '2024', '2025']):
        if row[f'{year} sbti nt'] == '1' and first_sbti_year is None:
            first_sbti_year = i
        if row[f'{year} nz'] == '1' and first_nz_year is None:
            first_nz_year = i
    
    if first_sbti_year is not None and first_nz_year is not None and first_sbti_year < first_nz_year:
        transitions.append(first_nz_year - first_sbti_year)

print(f"Companies that got SBTi before NZ: {len(transitions)}")
print(f"Average years from SBTi to NZ: {sum(transitions)/len(transitions):.1f}")
print(f"Distribution: {pd.Series(transitions).value_counts().sort_index()}")

Companies that got SBTi before NZ: 47
Average years from SBTi to NZ: 1.7
Distribution: 1    28
2     8
3     7
4     4
Name: count, dtype: int64


In [17]:
import plotly.graph_objects as go

years_pairs = [('2021', '2023'), ('2023', '2025')]
nodes = []
node_colors = []

for year_pair in years_pairs:
    for year in year_pair:
        nodes.extend([f'{year} None', f'{year} SBTi only', f'{year} NZ only', f'{year} Both'])
        node_colors.extend(['#cccccc', '#008ACA', '#00B4B2', '#8B479B'])

source = []
target = []
value = []

for pair_idx, (start_year, end_year) in enumerate(years_pairs):
    mask = (portfolio[f'{start_year} sbti nt'] != '-') & (portfolio[f'{end_year} sbti nt'] != '-') & \
           (portfolio[f'{start_year} nz'] != '-') & (portfolio[f'{end_year} nz'] != '-')
    subset = portfolio[mask]
    
    transitions = subset.groupby([f'{start_year} sbti nt', f'{start_year} nz', 
                                   f'{end_year} sbti nt', f'{end_year} nz']).size().reset_index(name='count')
    
    for _, row in transitions.iterrows():
        sbti_start = int(row[f'{start_year} sbti nt'])
        nz_start = int(row[f'{start_year} nz'])
        sbti_end = int(row[f'{end_year} sbti nt'])
        nz_end = int(row[f'{end_year} nz'])
        
        src_state = 0 if (sbti_start + nz_start) == 0 else (1 if sbti_start == 1 and nz_start == 0 else (2 if sbti_start == 0 and nz_start == 1 else 3))
        tgt_state = 0 if (sbti_end + nz_end) == 0 else (1 if sbti_end == 1 and nz_end == 0 else (2 if sbti_end == 0 and nz_end == 1 else 3))
        
        source.append(pair_idx * 8 + src_state)
        target.append(pair_idx * 8 + 4 + tgt_state)
        value.append(row['count'])

fig = go.Figure(data=[go.Sankey(
    node=dict(pad=15, thickness=20, label=nodes, color=node_colors),
    link=dict(source=source, target=target, value=value)
)])

fig.update_layout(title_text="2-Year Transitions: 2021→2023 and 2023→2025", height=600)
fig.write_html(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\sbti_nz_2year.html')
fig.show()

In [18]:
import plotly.graph_objects as go

years = ['2023', '2024', '2025']
nodes = []
node_colors = []

for year in years:
    nodes.extend([f'{year} None', f'{year} SBTi only', f'{year} NZ only', f'{year} Both'])
    node_colors.extend(['#cccccc', '#008ACA', '#00B4B2', '#8B479B'])

source = []
target = []
value = []

for i in range(len(years) - 1):
    current_year = years[i]
    next_year = years[i + 1]
    
    mask = (portfolio[f'{current_year} sbti nt'] != '-') & (portfolio[f'{next_year} sbti nt'] != '-') & \
           (portfolio[f'{current_year} nz'] != '-') & (portfolio[f'{next_year} nz'] != '-')
    subset = portfolio[mask]
    
    transitions = subset.groupby([f'{current_year} sbti nt', f'{current_year} nz', 
                                   f'{next_year} sbti nt', f'{next_year} nz']).size().reset_index(name='count')
    
    for _, row in transitions.iterrows():
        sbti_curr = int(row[f'{current_year} sbti nt'])
        nz_curr = int(row[f'{current_year} nz'])
        sbti_next = int(row[f'{next_year} sbti nt'])
        nz_next = int(row[f'{next_year} nz'])
        
        src_state = 0 if (sbti_curr + nz_curr) == 0 else (1 if sbti_curr and not nz_curr else (2 if nz_curr and not sbti_curr else 3))
        tgt_state = 0 if (sbti_next + nz_next) == 0 else (1 if sbti_next and not nz_next else (2 if nz_next and not sbti_next else 3))
        
        source.append(i * 4 + src_state)
        target.append((i + 1) * 4 + tgt_state)
        value.append(row['count'])

fig = go.Figure(data=[go.Sankey(
    node=dict(pad=15, thickness=20, label=nodes, color=node_colors),
    link=dict(source=source, target=target, value=value)
)])

fig.update_layout(title_text="SBTi & Net Zero Transitions: 2023→2024→2025", height=600)
fig.write_html(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\sbti_nz_yearly.html')
fig.show()

In [19]:
import plotly.graph_objects as go

years = ['2023', '2024', '2025']
nodes = []
node_colors = []

for year in years:
    nodes.extend([f'{year} None', f'{year} SBTi only', f'{year} NZ only', f'{year} Both'])
    node_colors.extend(['#cccccc', '#008ACA', '#00B4B2', '#8B479B'])

source = []
target = []
value = []

for i in range(len(years) - 1):
    curr_year = years[i]
    next_year = years[i + 1]
    
    transitions = portfolio.groupby([f'{curr_year} sbti nt', f'{curr_year} nz', 
                                      f'{next_year} sbti nt', f'{next_year} nz']).size().reset_index(name='count')
    
    for _, row in transitions.iterrows():
        if row[f'{curr_year} sbti nt'] == '-' or row[f'{next_year} sbti nt'] == '-':
            continue
        if row[f'{curr_year} nz'] == '-' or row[f'{next_year} nz'] == '-':
            continue
            
        sbti_c, nz_c = row[f'{curr_year} sbti nt'], row[f'{curr_year} nz']
        sbti_n, nz_n = row[f'{next_year} sbti nt'], row[f'{next_year} nz']
        
        src_state = 0 if sbti_c=='0' and nz_c=='0' else (1 if sbti_c=='1' and nz_c=='0' else (2 if sbti_c=='0' and nz_c=='1' else 3))
        tgt_state = 0 if sbti_n=='0' and nz_n=='0' else (1 if sbti_n=='1' and nz_n=='0' else (2 if sbti_n=='0' and nz_n=='1' else 3))
        
        source.append(i * 4 + src_state)
        target.append((i + 1) * 4 + tgt_state)
        value.append(row['count'])

fig = go.Figure(data=[go.Sankey(
    node=dict(pad=15, thickness=20, label=nodes, color=node_colors),
    link=dict(source=source, target=target, value=value)
)])

fig.update_layout(title_text="SBTi & Net Zero Transitions: 2023→2024→2025", height=600)
fig.write_html(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\sbti_nz_yearly.html')
fig.show()

In [20]:
print("2023 distribution:")
print(f"None: {((portfolio['2023 sbti nt']=='0') & (portfolio['2023 nz']=='0')).sum()}")
print(f"SBTi only: {((portfolio['2023 sbti nt']=='1') & (portfolio['2023 nz']=='0')).sum()}")
print(f"NZ only: {((portfolio['2023 sbti nt']=='0') & (portfolio['2023 nz']=='1')).sum()}")
print(f"Both: {((portfolio['2023 sbti nt']=='1') & (portfolio['2023 nz']=='1')).sum()}")

print("\n2024 distribution:")
print(f"None: {((portfolio['2024 sbti nt']=='0') & (portfolio['2024 nz']=='0')).sum()}")
print(f"SBTi only: {((portfolio['2024 sbti nt']=='1') & (portfolio['2024 nz']=='0')).sum()}")
print(f"NZ only: {((portfolio['2024 sbti nt']=='0') & (portfolio['2024 nz']=='1')).sum()}")
print(f"Both: {((portfolio['2024 sbti nt']=='1') & (portfolio['2024 nz']=='1')).sum()}")

print("\n2025 distribution:")
print(f"None: {((portfolio['2025 sbti nt']=='0') & (portfolio['2025 nz']=='0')).sum()}")
print(f"SBTi only: {((portfolio['2025 sbti nt']=='1') & (portfolio['2025 nz']=='0')).sum()}")
print(f"NZ only: {((portfolio['2025 sbti nt']=='0') & (portfolio['2025 nz']=='1')).sum()}")
print(f"Both: {((portfolio['2025 sbti nt']=='1') & (portfolio['2025 nz']=='1')).sum()}")

2023 distribution:
None: 0
SBTi only: 0
NZ only: 0
Both: 78

2024 distribution:
None: 0
SBTi only: 45
NZ only: 0
Both: 92

2025 distribution:
None: 206
SBTi only: 41
NZ only: 137
Both: 116


In [21]:
import plotly.graph_objects as go

# Get companies present in 2023
companies_2023 = portfolio[(portfolio['2023 sbti nt'] != '-') & (portfolio['2023 nz'] != '-')]['company'].tolist()

# Filter to only these companies across all years
subset = portfolio[portfolio['company'].isin(companies_2023)]

years = ['2023', '2024', '2025']
nodes = []
node_colors = []

for year in years:
    nodes.extend([f'{year} None', f'{year} SBTi only', f'{year} NZ only', f'{year} Both'])
    node_colors.extend(['#cccccc', '#008ACA', '#00B4B2', '#8B479B'])

source = []
target = []
value = []

for i in range(len(years) - 1):
    curr_year = years[i]
    next_year = years[i + 1]
    
    transitions = subset.groupby([f'{curr_year} sbti nt', f'{curr_year} nz', 
                                   f'{next_year} sbti nt', f'{next_year} nz']).size().reset_index(name='count')
    
    for _, row in transitions.iterrows():
        sbti_c, nz_c = row[f'{curr_year} sbti nt'], row[f'{curr_year} nz']
        sbti_n, nz_n = row[f'{next_year} sbti nt'], row[f'{next_year} nz']
        
        if sbti_c == '-' or nz_c == '-' or sbti_n == '-' or nz_n == '-':
            continue
        
        src_state = 0 if sbti_c=='0' and nz_c=='0' else (1 if sbti_c=='1' and nz_c=='0' else (2 if sbti_c=='0' and nz_c=='1' else 3))
        tgt_state = 0 if sbti_n=='0' and nz_n=='0' else (1 if sbti_n=='1' and nz_n=='0' else (2 if sbti_n=='0' and nz_n=='1' else 3))
        
        source.append(i * 4 + src_state)
        target.append((i + 1) * 4 + tgt_state)
        value.append(row['count'])

fig = go.Figure(data=[go.Sankey(
    node=dict(pad=15, thickness=20, label=nodes, color=node_colors),
    link=dict(source=source, target=target, value=value)
)])

fig.update_layout(title_text=f"Tracking {len(companies_2023)} companies from 2023→2024→2025", height=600)
fig.write_html(r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\sbti_nz_tracked.html')
fig.show()

In [24]:
import plotly.graph_objects as go
import pandas as pd

# Force fresh read
file_path = r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\progression anlysis data.xlsx'
portfolio = pd.read_excel(file_path, header=0)

def normalize(val):
    if pd.isna(val): return '-'
    if val == 0 or val == 'N': return '0'
    return '1'

for col in portfolio.columns:
    if 'sbti' in col or 'nz' in col or 'cn' in col or 're' in col or 'cc' in col:
        portfolio[col] = portfolio[col].apply(normalize)

companies_2023 = portfolio[(portfolio['2023 sbti nt'] != '-') & (portfolio['2023 nz'] != '-')]['company'].tolist()
subset = portfolio[portfolio['company'].isin(companies_2023)].copy()

years = ['2023', '2024', '2025']
nodes = []
node_colors = []

for year in years:
    nodes.extend([f'{year} None', f'{year} SBTi only', f'{year} NZ only', f'{year} Both'])
    node_colors.extend(['#cccccc', '#008ACA', '#00B4B2', '#8B479B'])

source = []
target = []
value = []

for i in range(len(years) - 1):
    curr_year = years[i]
    next_year = years[i + 1]
    
    transitions = subset.groupby([f'{curr_year} sbti nt', f'{curr_year} nz', f'{next_year} sbti nt', f'{next_year} nz']).size().reset_index(name='count')
    
    for _, row in transitions.iterrows():
        sbti_c, nz_c = row[f'{curr_year} sbti nt'], row[f'{curr_year} nz']
        sbti_n, nz_n = row[f'{next_year} sbti nt'], row[f'{next_year} nz']
        
        if '-' in [sbti_c, nz_c, sbti_n, nz_n]:
            continue
        
        src = 0 if (sbti_c=='0' and nz_c=='0') else (1 if sbti_c=='1' and nz_c=='0' else (2 if nz_c=='1' and sbti_c=='0' else 3))
        tgt = 0 if (sbti_n=='0' and nz_n=='0') else (1 if sbti_n=='1' and nz_n=='0' else (2 if nz_n=='1' and sbti_n=='0' else 3))
        
        source.append(i * 4 + src)
        target.append((i + 1) * 4 + tgt)
        value.append(row['count'])

fig = go.Figure(data=[go.Sankey(node=dict(pad=15, thickness=20, label=nodes, color=node_colors), link=dict(source=source, target=target, value=value))])
fig.update_layout(title_text=f"Tracking {len(companies_2023)} companies from 2023", height=600)
fig.show()

In [1]:
import plotly.graph_objects as go
import pandas as pd

file_path = r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\progression anlysis data.xlsx'
portfolio = pd.read_excel(file_path, header=0, engine='openpyxl')

def normalize(val):
    if pd.isna(val):
        return '-'
    if val in [0, 'N', '']:
        return '0'
    if val in [1, 'Y']:
        return '1'
    return str(val)

for col in portfolio.columns:
    if any(x in str(col).lower() for x in ['sbti', 'nz', 'cn', 're', 'cc']):
        portfolio[col] = portfolio[col].apply(normalize)

companies_2023 = portfolio[(portfolio['2023 sbti nt'] != '-') & (portfolio['2023 nz'] != '-')]['company'].tolist()
subset = portfolio[portfolio['company'].isin(companies_2023)]

years = ['2023', '2024', '2025']
nodes = []
node_colors = []

for year in years:
    nodes.extend([f'{year} None', f'{year} SBTi only', f'{year} NZ only', f'{year} Both'])
    node_colors.extend(['#cccccc', '#008ACA', '#00B4B2', '#8B479B'])

source = []
target = []
value = []

for i in range(len(years) - 1):
    curr = years[i]
    next = years[i + 1]
    
    trans = subset.groupby([f'{curr} sbti nt', f'{curr} nz', f'{next} sbti nt', f'{next} nz']).size().reset_index(name='count')
    
    for _, row in trans.iterrows():
        if '-' in [row[f'{curr} sbti nt'], row[f'{curr} nz'], row[f'{next} sbti nt'], row[f'{next} nz']]:
            continue
        
        s_curr = row[f'{curr} sbti nt'] == '1'
        n_curr = row[f'{curr} nz'] == '1'
        s_next = row[f'{next} sbti nt'] == '1'
        n_next = row[f'{next} nz'] == '1'
        
        src = 3 if (s_curr and n_curr) else (1 if s_curr else (2 if n_curr else 0))
        tgt = 3 if (s_next and n_next) else (1 if s_next else (2 if n_next else 0))
        
        source.append(i * 4 + src)
        target.append((i + 1) * 4 + tgt)
        value.append(row['count'])

fig = go.Figure(data=[go.Sankey(
    node=dict(pad=15, thickness=20, label=nodes, color=node_colors),
    link=dict(source=source, target=target, value=value)
)])
fig.update_layout(title_text=f"{len(companies_2023)} companies tracked 2023→2025", height=600)
fig.show()

In [2]:
import plotly.graph_objects as go
import pandas as pd

file_path = r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\progression anlysis data.xlsx'
portfolio = pd.read_excel(file_path, header=0, engine='openpyxl')

def normalize(val):
    if pd.isna(val):
        return '-'
    if val in [0, 'N', '']:
        return '0'
    if val in [1, 'Y']:
        return '1'
    return str(val)

for col in portfolio.columns:
    if any(x in str(col).lower() for x in ['sbti', 'nz', 'cn', 're', 'cc']):
        portfolio[col] = portfolio[col].apply(normalize)

companies_2023 = portfolio[(portfolio['2023 sbti nt'] != '-') & (portfolio['2023 nz'] != '-')]['company'].tolist()
subset = portfolio[portfolio['company'].isin(companies_2023)]

years = ['2023', '2024', '2025']
nodes = []
node_colors = []

for year in years:
    nodes.extend([f'{year} None', f'{year} SBTi only', f'{year} NZ only', f'{year} Both'])
    node_colors.extend(['#cccccc', '#008ACA', '#00B4B2', '#8B479B'])

source = []
target = []
value = []

for i in range(len(years) - 1):
    curr = years[i]
    next = years[i + 1]
    
    trans = subset.groupby([f'{curr} sbti nt', f'{curr} nz', f'{next} sbti nt', f'{next} nz']).size().reset_index(name='count')
    
    for _, row in trans.iterrows():
        if '-' in [row[f'{curr} sbti nt'], row[f'{curr} nz'], row[f'{next} sbti nt'], row[f'{next} nz']]:
            continue
        
        s_curr = row[f'{curr} sbti nt'] == '1'
        n_curr = row[f'{curr} nz'] == '1'
        s_next = row[f'{next} sbti nt'] == '1'
        n_next = row[f'{next} nz'] == '1'
        
        src = 3 if (s_curr and n_curr) else (1 if s_curr else (2 if n_curr else 0))
        tgt = 3 if (s_next and n_next) else (1 if s_next else (2 if n_next else 0))
        
        source.append(i * 4 + src)
        target.append((i + 1) * 4 + tgt)
        value.append(row['count'])

fig = go.Figure(data=[go.Sankey(
    node=dict(pad=15, thickness=20, label=nodes, color=node_colors),
    link=dict(source=source, target=target, value=value)
)])
fig.update_layout(title_text=f"{len(companies_2023)} companies tracked 2023→2025", height=600)
fig.show()

In [3]:
# Companies present in both 2021 and 2025 (not missing)
common_companies = portfolio[
    (portfolio['2021 cn '] != '-') & 
    (portfolio['2021 nz'] != '-') & 
    (portfolio['2021 sbti nt'] != '-') &
    (portfolio['2025 cn '] != '-') & 
    (portfolio['2025 nz'] != '-') & 
    (portfolio['2025 sbti nt'] != '-')
]

total_common = len(common_companies)
print(f"Total companies in both 2021 and 2025: {total_common}")

# CN (no NZ) to NZ transition
cn_to_nz_count = len(common_companies[
    (common_companies['2021 cn '] == '1') & 
    (common_companies['2021 nz'] == '0') & 
    (common_companies['2025 nz'] == '1')
])

# SBTi (no NZ) to NZ transition  
sbti_to_nz_count = len(common_companies[
    (common_companies['2021 sbti nt'] == '1') & 
    (common_companies['2021 nz'] == '0') & 
    (common_companies['2025 nz'] == '1')
])

print(f"\nCN (no NZ) → NZ: {cn_to_nz_count} companies ({cn_to_nz_count/total_common*100:.1f}%)")
print(f"SBTi (no NZ) → NZ: {sbti_to_nz_count} companies ({sbti_to_nz_count/total_common*100:.1f}%)")

Total companies in both 2021 and 2025: 10

CN (no NZ) → NZ: 0 companies (0.0%)
SBTi (no NZ) → NZ: 0 companies (0.0%)


In [7]:
import pandas as pd

file_path = r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\historic new - Copy.xlsx'

# Read 2021 and 2025 sheets
y2021 = pd.read_excel(file_path, sheet_name='2021', header=1)
y2025 = pd.read_excel(file_path, sheet_name='2025', header=1)

# Get company names from first column
companies_2021 = set(y2021[y2021.columns[0]].dropna().astype(str).tolist())
companies_2025 = set(y2025[y2025.columns[0]].dropna().astype(str).tolist())

# Find companies in BOTH years
common_companies = companies_2021.intersection(companies_2025)

print(f"Companies in 2021: {len(companies_2021)}")
print(f"Companies in 2025: {len(companies_2025)}")
print(f"Companies in BOTH years: {len(common_companies)}")
print(common_companies)
# Now rebuild portfolio with only these common companies
portfolio_new = portfolio[portfolio['company'].isin(common_companies)]

print(f"\nPortfolio filtered to common companies: {len(portfolio_new)}")

Companies in 2021: 500
Companies in 2025: 501
Companies in BOTH years: 381
{'Caterpillar', 'Sumitomo', 'Country Garden Holdings', 'Dell Technologies', 'China Railway Engineering Group', 'Saint-Gobain', 'Royal Bank of Canada', 'Valero Energy', 'Industrial & Commercial Bank of China', 'Liberty Mutual Insurance Group', 'Bristol-Myers Squibb', 'ABB', 'Bank of America', 'Pertamina', 'Verizon Communications', 'Zhejiang Hengyi Group', 'KB Financial Group', 'Lloyds Banking Group', 'Performance Food Group', 'Taikang Insurance Group', 'Apple', 'Beijing Automotive Group', 'AEON', 'New York Life Insurance', 'Dai-ichi Life Holdings', 'Wuchan Zhongda Group', 'CRH', 'Mondelez International', 'Banco Bilbao Vizcaya Argentaria', 'Shanghai Construction Group', 'Nationwide', 'Vale', 'Idemitsu Kosan', 'Schneider Electric', 'China Mobile Communications', 'Goldman Sachs Group', 'George Weston', 'BMW Group', 'China Merchants Bank', 'Siemens Energy', 'Publix Super Markets', 'SAIC Motor', 'JD.com', 'Siemens', '

In [8]:
# Companies with CN in 2021
cn_2021 = portfolio_new[portfolio_new['2021 cn '] == '1']
print(f"Companies with CN in 2021: {len(cn_2021)}")
print(cn_2021[['company', '2021 cn ', '2021 nz', '2025 cn ', '2025 nz']])

# Companies with CN but NO NZ in 2021
cn_no_nz_2021 = portfolio_new[(portfolio_new['2021 cn '] == '1') & (portfolio_new['2021 nz'] == '0')]
print(f"\nCompanies with CN but NO NZ in 2021: {len(cn_no_nz_2021)}")
print(cn_no_nz_2021[['company', '2021 cn ', '2021 nz', '2025 cn ', '2025 nz']])

# Companies with NZ in 2025
nz_2025 = portfolio_new[portfolio_new['2025 nz'] == '1']
print(f"\nCompanies with NZ in 2025: {len(nz_2025)}")
print(nz_2025[['company', '2021 cn ', '2021 nz', '2025 cn ', '2025 nz']])

# The transition: CN (no NZ) in 2021 → NZ in 2025
cn_to_nz = portfolio_new[
    (portfolio_new['2021 cn '] == '1') & 
    (portfolio_new['2021 nz'] == '0') & 
    (portfolio_new['2025 nz'] == '1')
]
print(f"\nCompanies that went from CN (no NZ) to NZ: {len(cn_to_nz)}")
print(f"Percentage of common companies: {len(cn_to_nz)/len(portfolio_new)*100:.1f}%")
print(cn_to_nz[['company', '2021 cn ', '2021 nz', '2025 cn ', '2025 nz', 'country', 'sector ']])

Companies with CN in 2021: 121
                    company 2021 cn  2021 nz 2025 cn  2025 nz
1                       ABB        1       -        0       1
7                      AT&T        1       -        1       1
8                       AXA        1       1        1       1
22                  Allianz        1       1        1       1
24                 Alphabet        1       -        0       1
..                      ...      ...     ...      ...     ...
620                   Volvo        1       1        1       1
627             Wells Fargo        1       1        1       0
635         X5 Retail Group        1       -        0       0
644      ZF Friedrichshafen        1       -        0       0
650  Zurich Insurance Group        1       1        1       1

[121 rows x 5 columns]

Companies with CN but NO NZ in 2021: 0
Empty DataFrame
Columns: [company, 2021 cn , 2021 nz, 2025 cn , 2025 nz]
Index: []

Companies with NZ in 2025: 192
                    company 2021 cn  2021 nz 2

In [6]:
# Companies present in both 2021 and 2025 (not missing)
common_companies = portfolio[
    (portfolio['2021 cn '] != '-') & 
    (portfolio['2021 nz'] != '-') & 
    (portfolio['2021 sbti nt'] != '-') &
    (portfolio['2025 cn '] != '-') & 
    (portfolio['2025 nz'] != '-') & 
    (portfolio['2025 sbti nt'] != '-')
]

total_common = len(common_companies)
print(f"Total companies in both 2021 and 2025: {total_common}")

# CN (no NZ) to NZ transition
cn_to_nz_count = len(common_companies[
    (common_companies['2021 cn '] == '1') & 
    (common_companies['2021 nz'] == '0') & 
    (common_companies['2025 nz'] == '1')
])

# SBTi (no NZ) to NZ transition  
sbti_to_nz_count = len(common_companies[
    (common_companies['2021 sbti nt'] == '1') & 
    (common_companies['2021 nz'] == '0') & 
    (common_companies['2025 nz'] == '1')
])

print(f"\nCN (no NZ) → NZ: {cn_to_nz_count} companies ({cn_to_nz_count/total_common*100:.1f}%)")
print(f"SBTi (no NZ) → NZ: {sbti_to_nz_count} companies ({sbti_to_nz_count/total_common*100:.1f}%)")

Total companies in both 2021 and 2025: 10

CN (no NZ) → NZ: 0 companies (0.0%)
SBTi (no NZ) → NZ: 0 companies (0.0%)


In [9]:
# Companies with SBTi in 2021
sbti_2021 = portfolio_new[portfolio_new['2021 sbti nt'] == '1']
print(f"Companies with SBTi in 2021: {len(sbti_2021)}")
print(sbti_2021[['company', '2021 sbti nt', '2021 nz', '2025 sbti nt', '2025 nz']])

# Companies with SBTi but NO NZ in 2021
sbti_no_nz_2021 = portfolio_new[(portfolio_new['2021 sbti nt'] == '1') & (portfolio_new['2021 nz'] == '0')]
print(f"\nCompanies with SBTi but NO NZ in 2021: {len(sbti_no_nz_2021)}")
print(sbti_no_nz_2021[['company', '2021 sbti nt', '2021 nz', '2025 sbti nt', '2025 nz']])

# Companies with NZ in 2025
nz_2025 = portfolio_new[portfolio_new['2025 nz'] == '1']
print(f"\nCompanies with NZ in 2025: {len(nz_2025)}")
print(nz_2025[['company', '2021 sbti nt', '2021 nz', '2025 sbti nt', '2025 nz']])

# The transition: SBTi (no NZ) in 2021 → NZ in 2025
sbti_to_nz = portfolio_new[
    (portfolio_new['2021 sbti nt'] == '1') & 
    (portfolio_new['2021 nz'] == '0') & 
    (portfolio_new['2025 nz'] == '1')
]
print(f"\nCompanies that went from SBTi (no NZ) to NZ: {len(sbti_to_nz)}")
print(f"Percentage of common companies: {len(sbti_to_nz)/len(portfolio_new)*100:.1f}%")
print(sbti_to_nz[['company', '2021 sbti nt', '2021 nz', '2025 sbti nt', '2025 nz', 'country', 'sector ']])

Companies with SBTi in 2021: 68
                  company 2021 sbti nt 2021 nz 2025 sbti nt 2025 nz
1                     ABB            1       -            1       1
3                    AEON            1       1            1       1
7                    AT&T            1       -            1       1
11              Accenture            1       1            1       1
36   Anheuser-Busch InBev            1       -            1       1
..                    ...          ...     ...          ...     ...
618        Vodafone Group            1       1            1       1
619            Volkswagen            1       -            1       0
620                 Volvo            1       1            1       1
623               Walmart            1       1            1       1
631      Woolworths Group            1       1            1       1

[68 rows x 5 columns]

Companies with SBTi but NO NZ in 2021: 0
Empty DataFrame
Columns: [company, 2021 sbti nt, 2021 nz, 2025 sbti nt, 2025 nz]
Index: